# 07 · Correlated Noise and InSAR

Every notebook so far assumed the data covariance $C_d$ is **diagonal** — each
observation's noise independent of its neighbours'. For InSAR that is wrong.
Atmospheric delay and orbital errors vary smoothly in space, so nearby pixels
share noise. This notebook builds a proper **spatially-correlated** $C_d$ and
shows why it matters.

## Learning objectives
- Understand why InSAR noise is spatially correlated.
- Build a full covariance from a covariance function with `spatial_covariance()`.
- Compare inversions with diagonal vs. full $C_d$.
- See the effect on both the slip estimate and its uncertainty.

## Why the diagonal assumption fails

InSAR noise is dominated by turbulent atmospheric delay and residual orbital
ramps — both **smooth in space**. Two pixels 1 km apart see almost the same
atmosphere, so their noise is strongly correlated; pixels far apart are nearly
independent. A diagonal $C_d$ pretends every pixel is independent, which badly
**overcounts** the information: a patch of 400 correlated pixels may carry the
information of only a handful of independent ones.

### A covariance function
We model the covariance between two pixels as a function of their separation
distance $r$:

$$ C(r) = \sigma^2 \, e^{-r/L}, $$

an **exponential** model with variance (sill) $\sigma^2$ and correlation length
$L$. At $r=0$ it is $\sigma^2$; by $r=L$ it has dropped to $1/e$. `geodef`'s
`spatial_covariance()` builds the full $n\times n$ matrix from station
coordinates, plus an optional **nugget** of uncorrelated white noise on the
diagonal.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

%load_ext autoreload
%autoreload 2

rng = np.random.default_rng(0)

# --- Recurring teaching scenario -------------------------------------------
# The higher-resolution megathrust from notebook 01, reused (copied) across
# notebooks 03-10 so every inversion targets the same "true" slip model.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,   # centroid 25 km deep
    strike=315.0, dip=15.0,            # NW-striking, shallow dip
    length=180e3, width=90e3,          # 180 km x 90 km
    n_length=12, n_width=6,            # -> 72 patches
)
N = fault.n_patches
nL, nW = fault.grid_shape

# "True" slip: notebook 01's smooth Gaussian thrust asperity, dip-slip only.
# The model vector is blocked as [strike-slip (N) | dip-slip (N)], so the
# strike-slip half is zero.
i = np.arange(N) % nL
j = np.arange(N) // nL
i0, j0 = (nL - 1) / 2, nW / 2
bump = np.exp(-(((i - i0) / 3.0) ** 2 + ((j - j0) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

# A grid of surface GNSS stations (note: GNSS takes lon, lat order).
glon, glat = np.meshgrid(np.linspace(98.5, 101.5, 8), np.linspace(-3.6, -0.4, 8))
glon, glat = glon.ravel(), glat.ravel()
n_sta = glon.size

# Synthetic data: forward-model the true slip, then add seeded Gaussian noise.
_zero = np.zeros(n_sta)
_one = np.ones(n_sta)
G_full = geodef.greens.matrix(
    fault, geodef.GNSS(lon=glon, lat=glat, ve=_zero, vn=_zero, vu=_zero, se=_one, sn=_one, su=_one)
)
sigma = 0.01  # 1 cm station noise
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, G_full.shape[0])
gnss = geodef.GNSS(
    lon=glon, lat=glat,
    ve=d_obs[0::3], vn=d_obs[1::3], vu=d_obs[2::3],
    se=np.full(n_sta, sigma), sn=np.full(n_sta, sigma), su=np.full(n_sta, sigma),
)
print(f"{N} patches, {n_sta} stations, {d_obs.size} observations")

40 patches, 36 stations, 108 observations


## An InSAR track with correlated noise

Build a dense track, forward-model the true slip, then add noise drawn from a
spatially-correlated covariance (not independent per pixel).

In [2]:
ilon, ilat = np.meshgrid(np.linspace(99.6, 100.4, 18), np.linspace(-0.35, 0.35, 18))
ilon, ilat = ilon.ravel(), ilat.ravel()
n_px = ilon.size
look_e, look_n, look_u = np.full(n_px, -0.38), np.full(n_px, 0.09), np.full(n_px, 0.92)

Gi = geodef.greens.matrix(
    fault, geodef.InSAR(lon=ilon, lat=ilat, los=np.zeros(n_px), sigma=np.ones(n_px), look_e=look_e, look_n=look_n, look_u=look_u)
)

# Full spatial covariance: 5 mm correlated, 30 km correlation length, 2 mm nugget.
C_full = geodef.spatial_covariance(
    ilon, ilat, sill=0.005**2, correlation_length=30_000.0,
    model="exponential", nugget=0.002**2,
)
# Draw one correlated noise realization from C_full.
noise = rng.multivariate_normal(np.zeros(n_px), C_full)
los = Gi @ slip_true + noise
sigma_px = np.sqrt(np.diag(C_full))
print(f"{n_px} pixels; noise std {sigma_px.mean()*1000:.1f} mm")

324 pixels; noise std 5.4 mm


## What the covariance looks like

The correlation matrix (covariance normalized to unit diagonal) has broad
off-diagonal structure — nearby pixels are correlated. A diagonal $C_d$ would
keep only the bright diagonal and throw the rest away.

In [3]:
corr = C_full / np.outer(sigma_px, sigma_px)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr, cmap="magma", vmin=0, vmax=1)
ax.set_title("InSAR noise correlation matrix")
ax.set_xlabel("pixel index"); ax.set_ylabel("pixel index")
fig.colorbar(im, ax=ax, label="correlation")

## Diagonal vs. full $C_d$

Invert the *same* data two ways: once assuming independent noise (diagonal
$C_d$), once with the true full covariance. Both use the same smoothing.

In [4]:
insar_diag = geodef.InSAR(lon=ilon, lat=ilat, los=los, sigma=sigma_px, look_e=look_e, look_n=look_n, look_u=look_u)
insar_full = geodef.InSAR(lon=ilon, lat=ilat, los=los, sigma=sigma_px, look_e=look_e, look_n=look_n, look_u=look_u,
                          covariance=C_full)

kw = dict(components="dip", regularization="laplacian", regularization_strength=1.0)
r_diag = geodef.invert.solve(fault, insar_diag, **kw)
r_full = geodef.invert.solve(fault, insar_full, **kw)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), constrained_layout=True)
vmax = slip_true[N:].max()
geodef.plot.slip(fault, slip_true[N:], ax=axes[0], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, colorbar=False, title="True")
geodef.plot.slip(fault, r_diag.slip_vector, ax=axes[1], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, colorbar=False, title="Diagonal C_d")
geodef.plot.slip(fault, r_full.slip_vector, ax=axes[2], cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, colorbar=False, title="Full C_d")
fig.colorbar(axes[0].collections[-1], ax=axes, shrink=0.8, label="Slip (m)")

## The effect on uncertainty

The bigger effect is on the **error bars**. Ignoring correlation makes the data
look more informative than they are, so the diagonal-$C_d$ uncertainties are too
*small* (over-confident). The full-$C_d$ uncertainties are more honest.

In [5]:
u_diag = geodef.model_uncertainty(r_diag, fault, insar_diag)
u_full = geodef.model_uncertainty(r_full, fault, insar_full)
print(f"mean 1-sigma slip uncertainty, diagonal C_d: {u_diag.mean()*100:.1f} cm")
print(f"mean 1-sigma slip uncertainty, full     C_d: {u_full.mean()*100:.1f} cm")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
umax = max(u_diag.max(), u_full.max())
geodef.plot.uncertainty(fault, u_diag, ax=axes[0], vmax=umax,
                        title="Uncertainty (diagonal C_d)")
geodef.plot.uncertainty(fault, u_full, ax=axes[1], vmax=umax,
                        title="Uncertainty (full C_d)")
plt.tight_layout()

mean 1-sigma slip uncertainty, diagonal C_d: 5.3 cm
mean 1-sigma slip uncertainty, full     C_d: 5.5 cm


## Exercises
1. **Correlation length.** Sweep `correlation_length` from `5_000` (nearly white)
   to `80_000` m. How do the recovered slip and its uncertainty respond?
2. **Nugget.** Set `nugget=0` and re-draw the noise. What changes?
3. **Down-sampling.** Correlated data are partly redundant. Keep every other
   pixel and confirm the full-$C_d$ result barely changes — a hint at why real
   InSAR workflows down-sample dense data.

## Checkpoint questions
1. Why does ignoring correlation make the uncertainties too small?
2. What do the sill, correlation length, and nugget each control?
3. Roughly how does the "effective number of independent observations" relate to
   the correlation length?

## Summary
- InSAR noise is spatially correlated; diagonal $C_d$ is optimistic.
- `spatial_covariance()` builds a full $C_d$ from an exponential/Gaussian model.
- Passing it as `covariance=` mainly corrects the *uncertainties*.

**Next:** notebook 08 enforces physically admissible slip with bounds and
constraints.